# E2 · The production model does not match the validated one

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Find out why the probabilities computed with `model.get_booster().predict(DMatrix)` (used "in production", for explanations and for export) differ from `model.predict_proba` (used in validation) and why the per-example contributions **do not sum** to the margin of the scikit-learn wrapper: after early stopping the booster contains
**all** the trained trees, including the 30 added after the minimum, and the wrapper ignores them automatically while the native booster does not.

## Dataset

`make_classification(3000, 12 features, 5 informative, flip_y=0.1)`, 60/20/20 split (training / validation / test). `model = XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, early_stopping_rounds=30)` trained with the validation set in `eval_set`.

## Educational purpose

Know that `best_iteration` is an attribute of the model, **not** a truncation of the booster: after `fit` with early stopping the booster has `best_iteration + 31` trees. The wrapper `predict`/`predict_proba` uses only the first `best_iteration + 1`; `Booster.predict`, `pred_contribs` and (partly) saving do not, unless you tell them to.

## Task

1. Run the colleague's cell: `max_diff` (maximum absolute gap between `production_p` = `booster.predict(DMatrix(X_te))` and `validated_p` = `model.predict_proba(X_te)[:, 1]`) and `production_ll`, `validated_ll`.
2. Diagnosis: save `best` (`model.best_iteration`), `total_trees` (`booster.num_boosted_rounds()`) and `extra_trees` (`total_trees - (best + 1)`).
3. Fix: `ok_p = booster.predict(DMatrix(X_te), iteration_range=(0, best + 1))` and `ok_p_matches` (True if `ok_p` matches `validated_p`). Save `contribs` (`pred_contribs=True`, with the same `iteration_range`) and `sum_ok` (True if `contribs.sum(1)` matches `model.predict(X_te, output_margin=True)`).
4. Measure the damage of the wrong explanation: `bug_sum_gap` (maximum absolute gap between the sum of the contributions computed **without** `iteration_range` and the margin of the wrapper) and `top_changes` (True if the ranking of the most important feature by mean of the absolute values of the contributions changes between the wrong version and the correct one: it may also not change: answer with what you measure).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X, y = make_classification(3000, n_features=12, n_informative=5, class_sep=0.7, flip_y=0.1, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
booster = model.get_booster()
dte = xgb.DMatrix(X_te)

## Your work

### The colleague's code

In [ ]:
validated_p = model.predict_proba(X_te)[:, 1]                        # in validation
production_p = booster.predict(dte)                                  # 'in production we use the booster: it is faster'
max_diff = float(np.abs(production_p - validated_p).max())
validated_ll, production_ll = log_loss(y_te, validated_p), log_loss(y_te, production_p)
print(f"maximum gap between the probabilities: {max_diff:.4f} | validated log-loss {validated_ll:.4f}, in production {production_ll:.4f}")

### Your diagnosis and the fix

In [ ]:
best = None
total_trees = None
extra_trees = None

In [ ]:
ok_p = None
ok_p_matches = None
contribs = None
sum_ok = None

In [ ]:
bug_sum_gap = None
top_changes = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`booster.num_boosted_rounds()` counts the trees (one tree per round for binary classification). Margin of the wrapper: `model.predict(X_te, output_margin=True)`. Contributions with `iteration_range`: `booster.predict(dte, pred_contribs=True, iteration_range=(0, best + 1))`.'):
    check(max_diff > 0.03 and production_ll > validated_ll, "the colleague sees a gap of over 0.03 between the probabilities and a worse log-loss in production", "don't modify the colleague's code")
    check(best is not None and total_trees is not None and total_trees == best + 31 and extra_trees == 30, "the booster has best_iteration + 31 trees: the 30 rounds of patience are inside the model", "booster.num_boosted_rounds()")
    check(ok_p is not None and ok_p_matches is True, "with iteration_range=(0, best + 1) the booster reproduces predict_proba", "iteration_range=(0, best_iteration + 1)")
    check(contribs is not None and contribs.shape == (len(X_te), 13) and sum_ok is True, "13 columns (12 features and the bias) and sum equal to the margin of the wrapper", "pred_contribs=True with the same iteration_range")
    check(bug_sum_gap is not None and bug_sum_gap > 0.3 and top_changes in (True, False), "without iteration_range the sum of the contributions departs by over 0.3 from the validated margin", "sum of the contributions minus margin")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_the_booster_ignores_early_stopping.ipynb`. Open it after you pass the check, or after you have really tried.